# 07 — One agent becomes a Crew
The Researcher finds evidence, the Analyst weighs it, and the Critic checks
the conclusion. A sequential process passes task context along the chain.



## Problem

One agent may research, decide, and self-review without a clear handoff.


## Concept

A sequential Crew assigns evidence, analysis, and critique to specialists.

A Crew groups specialist Agents and Tasks. Sequential process plus explicit task context establishes a handoff; a structured final output makes the decision easier to validate.


## Abstraction

Researcher → Analyst → Critic → structured DecisionReport. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
question + customer context
          │
          ▼
Researcher (tools + knowledge) ──▶ evidence notes
          ▼
Analyst (ROI scenario) ──▶ conditional recommendation
          ▼
Critic (source check) ──▶ DecisionReport
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Define three CrewAI Agents with distinct roles, goals, and tool permissions

Define three CrewAI Agents with distinct roles, goals, and tool permissions.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task

from workshop.common import model
from workshop.knowledge import sample_knowledge
from workshop.schemas import DecisionReport
from workshop.tools import company_search_tool, research_tool, roi_tool

llm = model()
researcher = Agent(
    role="Enterprise AI Researcher",
    goal="Gather source-backed facts about the proposed technology",
    backstory="You distinguish evidence from assumptions.",
    llm=llm,
    tools=[company_search_tool, research_tool],
    max_iter=4,
    allow_delegation=False,
)
analyst = Agent(
    role="Enterprise AI Analyst",
    goal="Weigh benefits, costs, constraints, and unknowns",
    backstory="You make conditional, auditable recommendations.",
    llm=llm,
    tools=[roi_tool],
    max_iter=4,
    allow_delegation=False,
)
critic = Agent(
    role="Evidence Critic",
    goal="Find unsupported claims and decide what must be verified",
    backstory="You accept only claims supported by named sources.",
    llm=llm,
    max_iter=4,
    allow_delegation=False,
)



## Give every specialist a distinct task
`context` makes the handoff explicit. The final task asks CrewAI to parse a
DecisionReport, but the surrounding harness still validates its source IDs.



## Step 2 — Connect three Tasks with explicit context and require a typed final report

Connect three Tasks with explicit context and require a typed final report.


In [ ]:
research = Task(
    description="Research {question}. Customer constraint: {customer_context}. "
    "Use tools and name the source IDs for every factual claim.",
    expected_output="Evidence notes with source IDs and explicit unknowns.",
    agent=researcher,
)
analysis = Task(
    description="Analyze the research for {question}; label ROI as a scenario.",
    expected_output="Conditional recommendation with tradeoffs and unknowns.",
    agent=analyst,
    context=[research],
)
critique = Task(
    description="Review the analysis and produce a structured final decision. "
    "Cite only architecture.md, security.md, or pricing.md in evidence.",
    expected_output="DecisionReport with recommendation, rationale, evidence, risks, next_steps.",
    agent=critic,
    context=[research, analysis],
    output_pydantic=DecisionReport,
    guardrail_max_retries=1,
)
sources, embedder = sample_knowledge()
crew = Crew(
    agents=[researcher, analyst, critic],
    tasks=[research, analysis, critique],
    process=Process.sequential,
    knowledge_sources=sources,
    embedder=embedder,
)
print("Roles:", [agent.role for agent in crew.agents])
print("Tasks:", [task.description[:55] for task in crew.tasks])



## Run and inspect the structured output
Ask whether the extra roles improved the evidence and critique enough to
justify their extra model calls and latency.



## Step 3 — Kick off the sequential Crew and inspect the typed DecisionReport, not just the final prose

Kick off the sequential Crew and inspect the typed DecisionReport, not just the final prose.


In [ ]:
result = crew.kickoff(inputs={
    "question": "Should Northstar AI pilot Meridian AI?",
    "customer_context": "Customer A requires private VPC deployment.",
})
print(result.pydantic or result.raw)


## Execution flow

Researcher → Analyst → Critic → structured DecisionReport. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

More agents add latency and can repeat an unsupported claim. The Critic must check named sources and the final report must be validated outside the model.


## Production implication

Add a specialist only when its distinct role improves measurable quality. Each extra handoff adds latency and a chance to propagate a bad claim.


## Your turn

Compare the final report with the one-agent answer from lesson 02.

**Checkpoint:** The report contains a conditional recommendation, citations, risks, and next steps.
